# Form Options Color Generator

This notebook injects hex color codes into form option labels based on semantic analysis.

**Requirements:**
- Every option gets a color
- No duplicate colors within the same question
- Map-readable colors (dark, saturated)
- Minimal JSON formatting changes (only color field additions)

In [ ]:
# Cell 1: Imports and Configuration
import json
import os
import re
import colorsys
from pathlib import Path
from typing import Optional, List, Dict, Any, Tuple
from collections import defaultdict

# Configuration
SOURCE_DIR = Path("../../backend/source/forms")
PROD_FILE_PATTERN = "*.prod.json"
PRESERVE_EXISTING_COLORS = True
DRY_RUN = True  # Set to False to apply changes

print(f"Source directory: {SOURCE_DIR.resolve()}")
print(f"Dry run mode: {DRY_RUN}")

In [ ]:
# Cell 2: Color Palettes (Map-Readable - Dark & Saturated)

# Semantic colors for classified labels
SEMANTIC_COLORS = {
    "positive": "#64A73B",      # Green - Operational, Good, Yes
    "negative": "#e41a1c",      # Red - Issue, Poor, No
    "warning": "#d95f02",       # Dark Orange - Maintenance, Moderate
    "neutral": "#1f78b4",       # Dark Blue - Satisfactory, Other
    "info": "#6a3d9a",          # Purple - Informational
}

# Risk level specific colors
RISK_LEVEL_COLORS = {
    "no risk": "#64A73B",       # Green
    "low risk": "#1f78b4",      # Dark Blue
    "moderate risk": "#d95f02", # Dark Orange
    "medium risk": "#d95f02",   # Dark Orange
    "high risk": "#e41a1c",     # Red
}

# 20-color fallback palette (dark, saturated for map visibility)
# Based on ColorBrewer "Dark2" + "Set1" qualitative palettes
FALLBACK_PALETTE = [
    "#1b9e77",  # Teal
    "#d95f02",  # Dark Orange
    "#7570b3",  # Purple
    "#e7298a",  # Magenta
    "#66a61e",  # Olive Green
    "#e6ab02",  # Dark Yellow/Gold
    "#a6761d",  # Brown
    "#666666",  # Dark Gray
    "#e41a1c",  # Red
    "#377eb8",  # Blue
    "#4daf4a",  # Green
    "#984ea3",  # Purple
    "#ff7f00",  # Orange
    "#a65628",  # Brown
    "#f781bf",  # Pink
    "#999999",  # Gray
    "#66c2a5",  # Cyan
    "#8da0cb",  # Steel Blue
    "#e78ac3",  # Light Pink
    "#a6d854",  # Lime
]

print(f"Semantic colors: {len(SEMANTIC_COLORS)}")
print(f"Fallback palette: {len(FALLBACK_PALETTE)} colors")

In [ ]:
# Cell 3: Classification Patterns

# Exact match mappings (highest priority)
EXACT_MATCH_MAP = {
    # Positive (Green)
    "yes": "positive",
    "operational": "positive",
    "good": "positive",
    "available": "positive",
    "completed": "positive",
    "secured": "positive",
    "covered": "positive",
    "fine": "positive",
    "normal": "positive",
    "normal operation": "positive",
    "oprational": "positive",  # Typo in data
    
    # Negative (Red)
    "no": "negative",
    "poor": "negative",
    "broken": "negative",
    "blocked": "negative",
    "missing": "negative",
    "faulty": "negative",
    "not available": "negative",
    "not operational": "negative",
    "non operational": "negative",
    "non-operational": "negative",
    "issue with the system": "negative",
    "not used": "negative",
    "not active": "negative",
    
    # Warning (Dark Orange)
    "satisfactory": "warning",
    "maintenance in progress": "warning",
    "intermittent": "warning",
    "ongoing": "warning",
    "need maintenance": "warning",
    "needs upgrade": "warning",
    
    # Neutral (Dark Blue)
    "other": "neutral",
    "others": "neutral",
}

# Negative patterns (partial match) - check first to catch problems
NEGATIVE_PATTERNS = [
    "failure", "failed", "problem", "issue", "error",
    "not ", "dead ", "hazard", "inadequate", "improper",
    "excessive", "overflow", "clogged", "clogging", "damaged",
    "leaking", "leakage", "defunct", "flooded",
    "g0 ",  # G0 = No toilet (sanitation negative)
]

# Positive patterns (partial match)
POSITIVE_PATTERNS = [
    "in operation", "working", "functional", "active",
    "g1 ",  # G1 = Toilet present (sanitation positive)
]

# Warning patterns (partial match)
WARNING_PATTERNS = [
    "partial", "mild", "some", "in progress",
    "moderate", "medium", "upgrading",
]

print(f"Exact matches: {len(EXACT_MATCH_MAP)}")
print(f"Negative patterns: {len(NEGATIVE_PATTERNS)}")
print(f"Positive patterns: {len(POSITIVE_PATTERNS)}")
print(f"Warning patterns: {len(WARNING_PATTERNS)}")

In [ ]:
# Cell 4: Classification Function

def classify_label(label: str) -> Optional[str]:
    """
    Classify a label and return appropriate hex color.
    Returns None if no semantic classification applies.
    """
    label_lower = label.lower().strip()
    
    # 1. Risk levels (special handling)
    if "risk" in label_lower:
        for term, color in RISK_LEVEL_COLORS.items():
            if term in label_lower:
                return color
    
    # 2. Exact matches
    if label_lower in EXACT_MATCH_MAP:
        return SEMANTIC_COLORS[EXACT_MATCH_MAP[label_lower]]
    
    # 3. Negative patterns (priority - catch problems first)
    for pattern in NEGATIVE_PATTERNS:
        if pattern in label_lower:
            return SEMANTIC_COLORS["negative"]
    
    # 4. Positive patterns
    for pattern in POSITIVE_PATTERNS:
        if pattern in label_lower:
            return SEMANTIC_COLORS["positive"]
    
    # 5. Warning patterns
    for pattern in WARNING_PATTERNS:
        if pattern in label_lower:
            return SEMANTIC_COLORS["warning"]
    
    # No semantic match
    return None


# Test classification
test_labels = [
    "Operational", "Issue with the system", "Good", "Poor",
    "High Risk", "Low Risk", "Lab Test", "CBT Test"
]
print("Classification tests:")
for label in test_labels:
    color = classify_label(label)
    print(f"  '{label}' -> {color or 'FALLBACK'}")

In [ ]:
# Cell 5: Color Assignment with Uniqueness Guarantee

def generate_dark_color(seed: int) -> str:
    """
    Generate a dark, saturated color for map visibility.
    Uses HSL with constrained lightness (30-50%) and saturation (70-100%).
    """
    hue = (seed * 137.508) % 360  # Golden angle for distribution
    saturation = 0.7 + (seed % 3) * 0.1  # 70-90%
    lightness = 0.35 + (seed % 4) * 0.05  # 35-50%
    
    r, g, b = colorsys.hls_to_rgb(hue/360, lightness, saturation)
    return "#{:02x}{:02x}{:02x}".format(int(r*255), int(g*255), int(b*255))


def assign_colors_to_options(
    options: List[Dict],
    preserve_existing: bool = True
) -> Tuple[List[Dict], Dict]:
    """
    Assign colors to all options ensuring:
    1. Semantic colors for classified labels
    2. NO duplicate colors within the same question
    3. Fallback colors for unclassified labels
    
    Returns:
        - Modified options list
        - Statistics dictionary
    """
    stats = {
        "total": len(options),
        "semantic": 0,
        "fallback": 0,
        "preserved": 0,
        "changes": []
    }
    
    used_colors = set()
    fallback_index = 0
    
    # First pass: collect existing colors
    if preserve_existing:
        for option in options:
            if option.get("color"):
                used_colors.add(option["color"].lower())
    
    # Second pass: assign colors
    for option in options:
        label = option.get("label", "")
        
        # Skip if already has color and preserving
        if preserve_existing and option.get("color"):
            stats["preserved"] += 1
            continue
        
        # Try semantic classification
        semantic_color = classify_label(label)
        assigned_color = None
        color_type = None
        
        if semantic_color and semantic_color.lower() not in used_colors:
            assigned_color = semantic_color
            color_type = "semantic"
            stats["semantic"] += 1
        else:
            # Use fallback palette
            while fallback_index < len(FALLBACK_PALETTE):
                candidate = FALLBACK_PALETTE[fallback_index]
                fallback_index += 1
                if candidate.lower() not in used_colors:
                    assigned_color = candidate
                    color_type = "fallback"
                    stats["fallback"] += 1
                    break
            
            # Generate if palette exhausted
            if not assigned_color:
                assigned_color = generate_dark_color(len(used_colors))
                color_type = "generated"
                stats["fallback"] += 1
        
        # Assign color
        option["color"] = assigned_color
        used_colors.add(assigned_color.lower())
        
        stats["changes"].append({
            "label": label,
            "color": assigned_color,
            "type": color_type
        })
    
    return options, stats


# Test uniqueness
test_options = [
    {"label": "Yes"},
    {"label": "No"},
    {"label": "Maybe"},
]
result, stats = assign_colors_to_options(test_options, preserve_existing=False)
print("Uniqueness test:")
for opt in result:
    print(f"  '{opt['label']}' -> {opt['color']}")
print(f"Stats: {stats['semantic']} semantic, {stats['fallback']} fallback")

In [ ]:
# Cell 6: JSON Processing (Minimal Formatting Changes)

def process_question(question: Dict, preserve_existing: bool = True) -> Tuple[Dict, Dict]:
    """Process a single question and assign colors to its options."""
    stats = {
        "question_id": question.get("id"),
        "question_name": question.get("name"),
        "question_label": question.get("label"),
        "has_options": False,
        "option_stats": None
    }
    
    if "options" in question and question["options"]:
        stats["has_options"] = True
        question["options"], stats["option_stats"] = assign_colors_to_options(
            question["options"],
            preserve_existing
        )
    
    return question, stats


def process_form(form_data: Dict, preserve_existing: bool = True) -> Tuple[Dict, Dict]:
    """Process entire form, assigning colors to all question options."""
    form_stats = {
        "form_id": form_data.get("id"),
        "form_name": form_data.get("form"),
        "questions_with_options": 0,
        "total_options": 0,
        "semantic_assignments": 0,
        "fallback_assignments": 0,
        "preserved_colors": 0,
        "question_details": []
    }
    
    for qg in form_data.get("question_groups", []):
        for question in qg.get("questions", []):
            question, q_stats = process_question(question, preserve_existing)
            
            if q_stats["option_stats"]:
                form_stats["questions_with_options"] += 1
                form_stats["total_options"] += q_stats["option_stats"]["total"]
                form_stats["semantic_assignments"] += q_stats["option_stats"]["semantic"]
                form_stats["fallback_assignments"] += q_stats["option_stats"]["fallback"]
                form_stats["preserved_colors"] += q_stats["option_stats"]["preserved"]
                form_stats["question_details"].append(q_stats)
    
    return form_data, form_stats

print("Processing functions defined.")

In [ ]:
# Cell 7: File I/O with Minimal Formatting Changes (Surgical Approach)

def load_json_file(filepath: Path) -> Tuple[Dict, str]:
    """
    Load JSON file and return data with original content.
    """
    with open(filepath, 'r', encoding='utf-8') as f:
        original_content = f.read()
    data = json.loads(original_content)
    return data, original_content


def inject_color_into_option(option_text: str, color: str) -> str:
    """
    Inject color field into an option object string.
    Preserves original formatting by inserting before the closing brace.
    """
    # Find the last property before closing brace
    # Match pattern: "key": value } or "key": value, "key2": value }
    
    # Remove trailing whitespace and closing brace
    option_text = option_text.rstrip()
    if option_text.endswith('}'):
        option_text = option_text[:-1].rstrip()
    
    # Check if it already has a trailing comma
    needs_comma = not option_text.rstrip().endswith(',')
    
    # Detect indentation from the option text
    lines = option_text.split('\n')
    if len(lines) > 1:
        # Multi-line option - get indentation from a property line
        for line in lines[1:]:
            stripped = line.lstrip()
            if stripped.startswith('"'):
                indent = line[:len(line) - len(stripped)]
                break
        else:
            indent = "              "  # Default 14 spaces
    else:
        # Single-line option - just add inline
        if needs_comma:
            return option_text + f', "color": "{color}" }}'
        else:
            return option_text + f' "color": "{color}" }}'
    
    # Multi-line: add color on new line with proper indentation
    if needs_comma:
        return option_text + f',\n{indent}"color": "{color}"\n{indent[:-2]}}}'
    else:
        return option_text + f'\n{indent}"color": "{color}"\n{indent[:-2]}}}'


def save_json_with_colors(
    filepath: Path, 
    original_content: str, 
    color_map: Dict[int, str]
) -> str:
    """
    Save JSON by surgically injecting colors into the original content.
    
    Args:
        filepath: Path to save
        original_content: Original file content
        color_map: Dict mapping option IDs to colors {option_id: color_hex}
    
    Returns:
        Modified content
    """
    content = original_content
    
    # For each option that needs a color, find it by ID and inject color
    for option_id, color in color_map.items():
        # Pattern to find option object with this ID that doesn't have color
        # Matches: { ... "id": 12345 ... } where there's no "color" field
        
        # First, find the option by its ID
        id_pattern = rf'"id":\s*{option_id}'
        id_match = re.search(id_pattern, content)
        
        if not id_match:
            continue
            
        # Find the enclosing object braces
        start_pos = id_match.start()
        
        # Search backwards for opening brace
        brace_count = 0
        obj_start = start_pos
        for i in range(start_pos, -1, -1):
            if content[i] == '}':
                brace_count += 1
            elif content[i] == '{':
                if brace_count == 0:
                    obj_start = i
                    break
                brace_count -= 1
        
        # Search forwards for closing brace
        brace_count = 0
        obj_end = start_pos
        for i in range(start_pos, len(content)):
            if content[i] == '{':
                brace_count += 1
            elif content[i] == '}':
                if brace_count == 0:
                    obj_end = i + 1
                    break
                brace_count -= 1
        
        option_text = content[obj_start:obj_end]
        
        # Skip if already has color
        if '"color"' in option_text:
            continue
        
        # Inject color before the closing brace
        # Find the last non-whitespace before }
        inner = option_text[1:-1].rstrip()  # Remove { and }
        
        # Detect if we're in compact or expanded format
        if '\n' in option_text:
            # Expanded format - add color on new line
            # Find indentation level
            lines = option_text.split('\n')
            indent = ""
            for line in lines:
                stripped = line.lstrip()
                if stripped.startswith('"'):
                    indent = line[:len(line) - len(stripped)]
                    break
            
            # Build new option
            if inner.endswith(','):
                inner = inner[:-1]
            
            # Find where to insert (before last closing brace line)
            last_brace_idx = option_text.rfind('}')
            before_brace = option_text[:last_brace_idx].rstrip()
            after_brace = option_text[last_brace_idx:]
            
            if not before_brace.endswith(','):
                before_brace += ','
            
            new_option = before_brace + f'\n{indent}"color": "{color}"' + '\n' + indent[:-2] + '}'
        else:
            # Compact format - add inline
            inner = inner.rstrip()
            if inner.endswith(','):
                inner = inner[:-1]
            new_option = '{ ' + inner + f', "color": "{color}" }}'
        
        content = content[:obj_start] + new_option + content[obj_end:]
    
    # Write to file
    with open(filepath, 'w', encoding='utf-8') as f:
        f.write(content)
    
    return content


def validate_json_structure(data: Dict) -> bool:
    """Validate JSON structure for form_seeder.py compatibility."""
    required = ["id", "form", "question_groups"]
    return all(field in data for field in required)

print("File I/O functions defined (surgical approach).")

In [ ]:
# Cell 8: Main Processing Loop (with surgical JSON modification)

# Question types that should have colored options
OPTION_QUESTION_TYPES = {"option", "multiple_option"}


def build_color_map(form_data: Dict, preserve_existing: bool = True) -> Tuple[Dict[int, str], Dict]:
    """
    Process form and build a map of option_id -> color.
    
    Only processes questions with type "option" or "multiple_option".
    
    Returns:
        - color_map: Dict mapping option IDs to color hex codes
        - form_stats: Statistics dictionary
    """
    color_map = {}
    form_stats = {
        "form_id": form_data.get("id"),
        "form_name": form_data.get("form"),
        "questions_with_options": 0,
        "total_options": 0,
        "semantic_assignments": 0,
        "fallback_assignments": 0,
        "preserved_colors": 0,
        "question_details": []
    }
    
    for qg in form_data.get("question_groups", []):
        for question in qg.get("questions", []):
            # Only process option and multiple_option question types
            question_type = question.get("type", "")
            if question_type not in OPTION_QUESTION_TYPES:
                continue
            
            if "options" not in question or not question["options"]:
                continue
            
            options = question["options"]
            q_stats = {
                "question_id": question.get("id"),
                "question_name": question.get("name"),
                "question_label": question.get("label"),
                "question_type": question_type,
                "has_options": True,
                "option_stats": {
                    "total": len(options),
                    "semantic": 0,
                    "fallback": 0,
                    "preserved": 0,
                    "changes": []
                }
            }
            
            used_colors = set()
            fallback_index = 0
            
            # First pass: collect existing colors
            if preserve_existing:
                for option in options:
                    if option.get("color"):
                        used_colors.add(option["color"].lower())
            
            # Second pass: assign colors
            for option in options:
                label = option.get("label", "")
                option_id = option.get("id")
                
                # Skip if already has color and preserving
                if preserve_existing and option.get("color"):
                    q_stats["option_stats"]["preserved"] += 1
                    continue
                
                # Try semantic classification
                semantic_color = classify_label(label)
                assigned_color = None
                color_type = None
                
                if semantic_color and semantic_color.lower() not in used_colors:
                    assigned_color = semantic_color
                    color_type = "semantic"
                    q_stats["option_stats"]["semantic"] += 1
                else:
                    # Use fallback palette
                    while fallback_index < len(FALLBACK_PALETTE):
                        candidate = FALLBACK_PALETTE[fallback_index]
                        fallback_index += 1
                        if candidate.lower() not in used_colors:
                            assigned_color = candidate
                            color_type = "fallback"
                            q_stats["option_stats"]["fallback"] += 1
                            break
                    
                    # Generate if palette exhausted
                    if not assigned_color:
                        assigned_color = generate_dark_color(len(used_colors))
                        color_type = "generated"
                        q_stats["option_stats"]["fallback"] += 1
                
                # Add to color map
                if option_id and assigned_color:
                    color_map[option_id] = assigned_color
                    used_colors.add(assigned_color.lower())
                    
                    q_stats["option_stats"]["changes"].append({
                        "label": label,
                        "color": assigned_color,
                        "type": color_type
                    })
            
            form_stats["questions_with_options"] += 1
            form_stats["total_options"] += q_stats["option_stats"]["total"]
            form_stats["semantic_assignments"] += q_stats["option_stats"]["semantic"]
            form_stats["fallback_assignments"] += q_stats["option_stats"]["fallback"]
            form_stats["preserved_colors"] += q_stats["option_stats"]["preserved"]
            form_stats["question_details"].append(q_stats)
    
    return color_map, form_stats


def process_all_forms(
    source_dir: Path,
    pattern: str = "*.prod.json",
    preserve_existing: bool = True,
    dry_run: bool = True
) -> Dict:
    """
    Process all form JSON files with surgical color injection.
    
    Args:
        source_dir: Directory containing form JSON files
        pattern: Glob pattern for file selection
        preserve_existing: If True, don't overwrite existing colors
        dry_run: If True, don't write files
    
    Returns:
        Results dictionary with statistics
    """
    results = {
        "files_processed": 0,
        "files_modified": 0,
        "errors": [],
        "form_stats": [],
        "dry_run": dry_run
    }
    
    files = sorted(source_dir.glob(pattern))
    print(f"Found {len(files)} files matching '{pattern}'\n")
    
    for filepath in files:
        print(f"Processing: {filepath.name}")
        
        try:
            # Load
            form_data, original_content = load_json_file(filepath)
            
            # Validate
            if not validate_json_structure(form_data):
                results["errors"].append({
                    "file": filepath.name,
                    "error": "Invalid structure"
                })
                continue
            
            # Build color map
            color_map, form_stats = build_color_map(form_data, preserve_existing)
            
            results["form_stats"].append({
                "file": filepath.name,
                "stats": form_stats
            })
            results["files_processed"] += 1
            
            # Check for changes
            changes = len(color_map)
            if changes > 0:
                results["files_modified"] += 1
                
                if not dry_run:
                    save_json_with_colors(filepath, original_content, color_map)
                    print(f"  > Saved: {changes} colors assigned")
                else:
                    print(f"  [DRY RUN] Would assign {changes} colors")
            else:
                print(f"  No changes needed")
        
        except Exception as e:
            import traceback
            results["errors"].append({
                "file": filepath.name,
                "error": str(e),
                "traceback": traceback.format_exc()
            })
            print(f"  X Error: {e}")
    
    return results

print(f"Main processing function defined. Only processes: {OPTION_QUESTION_TYPES}")

In [ ]:
# Cell 9: Reporting Functions

def generate_report(results: Dict) -> None:
    """Generate summary report."""
    print("\n" + "=" * 60)
    print("COLOR ASSIGNMENT REPORT")
    print("=" * 60)
    
    mode = "DRY RUN" if results["dry_run"] else "APPLIED"
    print(f"\nMode: {mode}")
    print(f"Files processed: {results['files_processed']}")
    print(f"Files modified: {results['files_modified']}")
    
    if results["errors"]:
        print(f"\nErrors ({len(results['errors'])})")
        for err in results["errors"]:
            print(f"  - {err['file']}: {err['error']}")
    
    # Totals
    total_options = sum(fs["stats"]["total_options"] for fs in results["form_stats"])
    total_semantic = sum(fs["stats"]["semantic_assignments"] for fs in results["form_stats"])
    total_fallback = sum(fs["stats"]["fallback_assignments"] for fs in results["form_stats"])
    total_preserved = sum(fs["stats"]["preserved_colors"] for fs in results["form_stats"])
    
    print(f"\nSUMMARY:")
    print(f"  Total options: {total_options}")
    print(f"  Semantic colors: {total_semantic}")
    print(f"  Fallback colors: {total_fallback}")
    print(f"  Preserved: {total_preserved}")
    print("=" * 60)


def show_detailed_assignments(results: Dict, show_type: str = "all") -> None:
    """
    Show detailed color assignments.
    
    Args:
        show_type: "all", "semantic", or "fallback"
    """
    print("\n" + "=" * 60)
    print(f"DETAILED ASSIGNMENTS ({show_type.upper()})")
    print("=" * 60)
    
    for fs in results["form_stats"]:
        file_shown = False
        
        for q in fs["stats"]["question_details"]:
            if not q["option_stats"] or not q["option_stats"]["changes"]:
                continue
            
            changes = q["option_stats"]["changes"]
            
            # Filter by type
            if show_type != "all":
                changes = [c for c in changes if c["type"] == show_type]
            
            if not changes:
                continue
            
            if not file_shown:
                print(f"\n{fs['file']}:")
                file_shown = True
            
            q_label = q.get('question_label', 'N/A') or 'N/A'
            if len(q_label) > 50:
                print(f"  Q: {q_label[:50]}...")
            else:
                print(f"  Q: {q_label}")
            for c in changes:
                icon = "[S]" if c["type"] == "semantic" else "[F]"
                print(f"    {icon} '{c['label']}' -> {c['color']}")

print("Reporting functions defined.")

In [ ]:
# Cell 10: DRY RUN - Preview Changes

print("Running in DRY RUN mode...\n")

results = process_all_forms(
    source_dir=SOURCE_DIR,
    pattern=PROD_FILE_PATTERN,
    preserve_existing=PRESERVE_EXISTING_COLORS,
    dry_run=True
)

generate_report(results)

In [ ]:
# Cell 11: Show Semantic Assignments

show_detailed_assignments(results, show_type="semantic")

In [ ]:
# Cell 12: Show Fallback Assignments

show_detailed_assignments(results, show_type="fallback")

In [ ]:
# Cell 13: APPLY CHANGES (Uncomment to execute)

# CAUTION: This will modify the JSON files!
# Make sure to review the dry-run results first.

# Uncomment the following lines to apply changes:

print("Applying changes...\n")
results = process_all_forms(
    source_dir=SOURCE_DIR,
    pattern=PROD_FILE_PATTERN,
    preserve_existing=PRESERVE_EXISTING_COLORS,
    dry_run=False
)
generate_report(results)

print("To apply changes, uncomment the code above and run this cell.")